In [ ]:
import joblib
# dataframe with explanation/subject as columns
# along with the brain map in MNI space (stored as scipy.sparse.coo_array)
flatmaps_dataframe = joblib.load('gct_flatmaps_dataframe.pkl')
flatmaps_dataframe.head(30)

# Process and save generative causal testing flatmaps into an easily loadable dataframe

In [4]:
from neuro.flatmaps_helper import load_flatmaps
from cortex import mni
import cortex
import pandas as pd
import joblib
import numpy as np
import scipy
from tqdm import tqdm


# main load
normalize_flatmaps = False
gemv_flatmaps_dict_S02, gemv_flatmaps_dict_S03, gemv_flatmaps_dict_S02_timecourse, gemv_flatmaps_dict_S03_timecourse = load_flatmaps(
    normalize_flatmaps, load_timecourse=True)

In [6]:
# Ensure FSL (flirt) is available to the kernel for cortex.mni.transform_to_mni
import os
_FSLDIR = os.environ.get('FSLDIR', '/usr/local/fsl')
os.environ['FSLDIR'] = _FSLDIR
_fsl_bin = os.path.join(_FSLDIR, 'share', 'fsl', 'bin')
if _fsl_bin not in os.environ.get('PATH', '').split(os.pathsep):
    os.environ['PATH'] = _fsl_bin + os.pathsep + os.environ.get('PATH', '')
os.environ.setdefault('FSLOUTPUTTYPE', 'NIFTI_GZ')
_MNI_TEMPLATE = os.path.join(_FSLDIR, 'data', 'standard', 'MNI152_T1_2mm_brain.nii.gz')

# merge gemv_flatmaps_dict_S02 and gemv_flatmaps_dict_S03 into a single dataframe
# The columns should be the subject, the current key split into two columns, and then a final column which stores the entire array of values
def merge_flatmaps_dicts_to_df(flatmaps_dict_S02, flatmaps_dict_S03):
    records = []
    for subject, flatmaps_dict in zip(['S02', 'S03'], [flatmaps_dict_S02, flatmaps_dict_S03]):
        for key, arr in flatmaps_dict.items():
            question, idx = key
            records.append({
                'subject': subject,
                'question': question,
                'index': idx,
                'values': arr
            })
    df = pd.DataFrame.from_records(records)
    return df

gemv_flatmaps_df = merge_flatmaps_dicts_to_df(gemv_flatmaps_dict_S02, gemv_flatmaps_dict_S03)

SUBJECT_ALIAS_TO_PYCORTEX = {
    'S01': 'UTS01',
    'S02': 'UTS02',
    'S03': 'UTS03',
}

def to_pycortex_subject(subject):
    return SUBJECT_ALIAS_TO_PYCORTEX.get(subject, subject)

def convert_to_mni_space(flatmap_arr, subject='UTS01'):
    flatmap_vol = cortex.Volume(
        data=flatmap_arr.flatten(), subject=subject, xfmname=f'{subject}_auto')
    flatmap_to_mni_cached = cortex.db.get_mnixfm(subject, f'{subject}_auto')
    mni_vol = mni.transform_to_mni(
        flatmap_vol, flatmap_to_mni_cached, template=_MNI_TEMPLATE)
    mni_arr = mni_vol.get_fdata()  # the actual array, shape=(91, 109, 91)

    # convert to sparse array format
    mni_arr = scipy.sparse.coo_array(mni_arr)    
    return mni_arr

# replace each array in the df with the corresponding MNI-space array (use a progress bar)
for idx, row in tqdm(gemv_flatmaps_df.iterrows(), total=len(gemv_flatmaps_df)):
    subject = to_pycortex_subject(row['subject'])
    flatmap_arr = row['values']
    gemv_flatmaps_df.at[idx, 'values'] = convert_to_mni_space(flatmap_arr, subject=subject)

100%|██████████| 108/108 [01:54<00:00,  1.06s/it]


In [7]:
joblib.dump(gemv_flatmaps_df, 'gct_flatmaps_dataframe.pkl')

['gct_flatmaps_dataframe.pkl']

In [9]:
!du -sh gct_flatmaps_dataframe.pkl

972M	gct_flatmaps_dataframe.pkl


/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=1346762) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()
